<a href="https://colab.research.google.com/github/anishkr-sahu/Genai/blob/main/Chromadb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install chromadb
!pip -q install chromadb openai langchain tiktoken

In [ ]:
# version of chromadb
!pip show chromadb

In [ ]:
!wget -q (link of document)

In [ ]:
# unzip the file and create a file
!unzip -q new-articles.zip -d new_articles

In [ ]:
# setting up environment
import os

os.environ["openai_api_key"] = ""

In [ ]:
# import some libraries
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.llms import OpenAI
from langchain.document_loaders import DirectoryLoader
from langchain.document_loaders import TextLoader

In [ ]:
# load data
loader = DirectoryLoader("/content/new_articles/", glob="./*.txt", loader_cls= TextLoader)

In [ ]:
document = loader.load()
document

In [ ]:
# split into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
text = text_splitter.split_document(document)

In [ ]:
len(text)
text[1]

In [ ]:
# creating chromadb
from langchain import embeddings
persist_directory = 'db'

embedding = OpenAIEmbeddings()
vectordb = Chroma.from_documents(documents=text,
                                 embedding=embedding,
                                 persist_directory=persist_directory)

In [ ]:
# persist the db to disk
vectordb.persist()
vectordb = None

In [ ]:
# now we can load the persist database from disk, and use it as normal:
vectordb = Chroma(persist_directory=persist_directory,
                  embedding_function=embedding)

In [ ]:
# make a retriever
retriever = vectordb.as_retriever()

In [ ]:
docs = retriever.get_relevant_documents("How much money did microsoft raise?")

In [ ]:
docs

In [ ]:
len(docs)

In [ ]:
retriever = vectordb.as_retriever(search_type={"k": 2})

In [ ]:
retriever.search_type

In [ ]:
# make a chain
from langchain.chains import RetrievalQA

In [ ]:
llm=OpenAI()

In [ ]:
# create the chain to answer questions
qa_chain = RetrievalQA.from_chain_type(llm=OpenAI(),
                                       chain_type="stuff",
                                       retriever=retriever,
                                       return_source_documents=True)

In [ ]:
# cite sources
def process_llm_response(llm_response):
  print(llm_response['result'])
  print('\n\nSources: ')
  for source in llm_response["source_documents"]:
    print(source.metadata['source'])

In [ ]:
# full example
query = "How much did microsoft raise?"
llm_response = qa_chain(query)
llm_response
process_llm_response(llm_response)
llm_response

In [ ]:
# deleting
!zip -r db.zip ./db

In [ ]:
# to cleanup, you can delete teh collection
vectordb.delete_collection()
vectordb.persist()

# delete the directory
!rm -rf db/

In [ ]:
# starting again loading the db
!unzip db.zip